# Introduction to LLMs using OpenRouter API
OpenRouter is a unified interface for LLMs meaning you have one API key and you can choose several models. In this course we'll use models that are free to use.

As of current writing you'll get 50 api calls each day for free.

We will try several different free models in this lecture, all through the OpenRouter API

In [1]:
# Agent: the object that sends your questions to the AI model and returns the answer
from pydantic_ai import Agent

In [2]:
# Agent: the object that sends your questions to the AI model and returns the answer
from pydantic_ai import Agent

# OpenAIChatModel: a wrapper that talks to any service using OpenAI's chat format
from pydantic_ai.models.openai import OpenAIChatModel

# OpenAIProvider: holds the connection details (where to connect and how to log in)
from pydantic_ai.providers.openai import OpenAIProvider

import os  # lets us read environment variables

from dotenv import load_dotenv  # reads key=value pairs from a .env file

# Load the variables from your .env file (e.g. NVIDIA_API=...) into the environment.
# This keeps secret keys out of your code so you don't accidentally commit them to git.
load_dotenv()

# Nvidia's API uses the same "chat completion" format as OpenAI,
# so we can reuse the OpenAI classes to talk to Nvidia models.
model = OpenAIChatModel(
    "nvidia/nemotron-3.5-lightning-30b-a3b",  # model name; any model from build.nvidia.com works
    provider=OpenAIProvider(  # connection details for Nvidia
        base_url="https://integrate.api.nvidia.com/v1",  # Nvidia's API address
        api_key=os.environ["NVIDIA_API"],  # your secret key, read from the environment
    ),
)

# The agent wraps the model. Later you can give it instructions, tools, etc.
agent = Agent(
    model,
    system_prompt="You are a joking programming bot that will always ask questions back in a nerdy joke style",
)

prompt = "Why is it foggy in Göteborg right now?"

# Send a question to the model and wait for the answer.
# `await` pauses until the model replies. It works directly in a Jupyter notebook;
# in a normal .py script you would need asyncio.run(...) instead (see below).
result = await agent.run(prompt)

# result.output holds the model's text answer
print(result.output)

                 pydantic-ai v2.62.0 • Python 3.13.7

      / \        agent: agent • model: openai:nvidia/nemotron-3.5-lightning-30b-a3b • tools: 0
     /   \         capabilities: 0
   /___.___\
  /    |    \    observability: off — see every model and tool call live, with cost
/      |      \    set it up free with Pydantic Logfire: uvx logfire-cli wizard --task pydantic-ai
`---.._|_..---'    or use any OpenTelemetry backend: https://pydantic.dev/docs/ai/logfire/#otel

                 goes away once observability is on — or PYDANTIC_AI_NO_BANNER=1


Are you experiencing a localized mist phenomenon, or did you forget to account for the dew point overflow in your regional weather simulation? ⚙️


In [3]:
print(result.output)

Are you experiencing a localized mist phenomenon, or did you forget to account for the dew point overflow in your regional weather simulation? ⚙️


## Check the messages

In [4]:
result.all_messages()

[ModelRequest(parts=[SystemPromptPart(content='You are a joking programming bot that will always ask questions back in a nerdy joke style', timestamp=datetime.datetime(2026, 10, 28, 13, 16, 58, 274280, tzinfo=datetime.timezone.utc)), UserPromptPart(content='Why is it foggy in Göteborg right now?', timestamp=datetime.datetime(2026, 10, 28, 13, 16, 58, 274293, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 10, 28, 13, 16, 58, 274910, tzinfo=datetime.timezone.utc), run_id='01a182a8-1e00-7276-9081-a822074056d8', conversation_id='01a182a8-1e00-7276-9081-a823143b2276'),
 ModelResponse(parts=[ThinkingPart(content='Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User asks: "Why is it foggy in Göteborg right now?"\n   - Context: Göteborg (Gothenburg), Sweden\n   - Tone/Role: I need to respond as a "joking programming bot that will always ask questions back in a nerdy joke style"\n\n2.  **Deconstruct the Persona Requirements:**\n   - "joking programming bot"\

In [5]:
len(result.all_messages())

2

In [6]:
result.all_messages()[0]

ModelRequest(parts=[SystemPromptPart(content='You are a joking programming bot that will always ask questions back in a nerdy joke style', timestamp=datetime.datetime(2026, 10, 28, 13, 16, 58, 274280, tzinfo=datetime.timezone.utc)), UserPromptPart(content='Why is it foggy in Göteborg right now?', timestamp=datetime.datetime(2026, 10, 28, 13, 16, 58, 274293, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 10, 28, 13, 16, 58, 274910, tzinfo=datetime.timezone.utc), run_id='01a182a8-1e00-7276-9081-a822074056d8', conversation_id='01a182a8-1e00-7276-9081-a823143b2276')

In [7]:
result.all_messages()[0].parts[1].content

'Why is it foggy in Göteborg right now?'

In [8]:
agent._system_prompts

('You are a joking programming bot that will always ask questions back in a nerdy joke style',)

# Analyze tokens
Tokens are the basic units of text for LLMs used to process and generate language. It is how LLMs divide the text into smaller units, for simplicity you could see a word as a token. Tokens are also what you pay for when you use the APIs

In [9]:
print(result.usage)
print(result.usage.input_tokens, result.usage.output_tokens)

RunUsage(details={'reasoning_tokens': 722}, input_tokens=45, output_reasoning_tokens=722, output_tokens=755, requests=1)
45 755


In [10]:
prompt = result.all_messages()[0].parts[1].content

In [11]:
print(f"system_prompt_words = {len(agent._system_prompts[0].split())}")
print(f"user_prompt_words = {len(prompt.split())}")
print(f"output_words = {len(result.output.split(" "))}")

system_prompt_words = 17
user_prompt_words = 8
output_words = 24


## Basic prompt engineering
Let's do some prompt engineering to make the model output text in a certain structure.

we will cover more prompt engineering and prompt design in later chapter

In [12]:
chef_bot = Agent(
    model,
    system_prompt=(
        "You are a professional chef that is very good at making asian food and explaining in a pedagogical way how to create yummy dishes.",
        """
        Answer in the following format:
        
        Description 1-2 sentences of what the dish is
        
        List of ingredients in bullet points

        Short step by step description on how to make the dish
        """
    ),
)

result = await chef_bot.run("Give me 3 asian soups in a list", timeout=120)
result

AgentRunResult(output="**1. Vietnamese Phở Gà (Chicken Pho)**  \nA fragrant, clear broth noodle soup featuring tender chicken, rice noodles, and aromatic spices like star anise and cinnamon, traditionally served with fresh herbs and lime on the side.  \n- Chicken bones and carcass for broth  \n- Charred onion and ginger  \n- Star anise, cinnamon stick, cloves  \n- Fish sauce, rock sugar  \n- Rice noodles (bánh phở)  \n- Chicken breast, thinly sliced  \n- Fresh Thai basil, cilantro, green onions  \n- Lime wedges, bean sprouts  \n\n**How to make it:**  \n1. Roast onion and ginger in a dry pan until charred; blanch and rinse chicken bones.  \n2. Simmer bones with charred aromatics, star anise, cinnamon, cloves, fish sauce, and rock sugar in 2 liters of water for 1–2 hours, skimming foam regularly.  \n3. Strain the broth, keep warm.  \n4. Blanch chicken slices in hot broth just until opaque, then divide cooked rice noodles into bowls.  \n5. Ladle hot broth over noodles and chicken, garnish

In [13]:
print(result.output)

**1. Vietnamese Phở Gà (Chicken Pho)**  
A fragrant, clear broth noodle soup featuring tender chicken, rice noodles, and aromatic spices like star anise and cinnamon, traditionally served with fresh herbs and lime on the side.  
- Chicken bones and carcass for broth  
- Charred onion and ginger  
- Star anise, cinnamon stick, cloves  
- Fish sauce, rock sugar  
- Rice noodles (bánh phở)  
- Chicken breast, thinly sliced  
- Fresh Thai basil, cilantro, green onions  
- Lime wedges, bean sprouts  

**How to make it:**  
1. Roast onion and ginger in a dry pan until charred; blanch and rinse chicken bones.  
2. Simmer bones with charred aromatics, star anise, cinnamon, cloves, fish sauce, and rock sugar in 2 liters of water for 1–2 hours, skimming foam regularly.  
3. Strain the broth, keep warm.  
4. Blanch chicken slices in hot broth just until opaque, then divide cooked rice noodles into bowls.  
5. Ladle hot broth over noodles and chicken, garnish with bean sprouts, herbs, and lime wed

# Temperature
The higher the temperature, the more likely the model is more creative, while lower temperatures, it is closer to deterministic. However it won't be entirely deterministic, even at temperature 0.

In [14]:
from pydantic_ai import ModelSettings

agent = Agent(
    model,
    system_prompt="You are a children storyteller. Answer max 3 sentences.",
    model_settings=ModelSettings(temperature=0),
)

prompt = "A gray netherland rabbit jumps ..."

result = await agent.run(prompt, timeout=120)
print(result.output)

A gray Netherlands rabbit jumps high into the fluffy white clouds, chasing a buzzing butterfly. With twitching nose and long ears flapping, it hops toward a garden of crisp carrots and sweet lettuce. The sun smiles down as the rabbit finds the perfect spot to rest and dream of carrot clouds.


In [15]:
result = await agent.run(prompt, timeout=120)
print(result.output)

A gray Netherland rabbit jumps high into a field of soft clover, his long ears flapping gently in the breeze. He lands quietly on the cool grass, ready for a day of curious adventures. With a happy twitch of his nose, he hops off to explore the magical garden beyond.


In [16]:
agent.model_settings = ModelSettings(temperature=1.5)
result = await agent.run(prompt, timeout=120)
print(result.output)

A gray Netherlands Dwarf rabbit named Pip leapt over sparkling brooks and chirping crickets. Each hop made a soft "bink!" sound, delighting the flowers around him. He landed gently on a mossy hill, ready for his next adventure.


In [17]:
result = await agent.run(prompt, timeout=120)
print(result.output)

A gray Netherlands rabbit named Nibbles bounced across the velvety meadow, his long ears flapping like soft flags in the breeze. He joyfully leaped over sparkling brooks and fluffy dandelion puffs, chasing the whisper of butterflies. With every hop, the meadow came alive with color and laughter, proving that even the smallest hearts can hop the biggest dreams.


# Multimodal inputs (Will give an error)
**nvidia/nemotron-3.5-lightning-30b-a3b is a text-only model, so the endpoint rejects any request that contains an image**

inputting an image and text
make sure to choose a model that can handle image input

# Do not run
from pathlib import Path
from pydantic_ai.messages import BinaryContent

image_joker = Agent(
    model,
    system_prompt="You are a person that can analyze an image, explain the contents and make a joke about it, max 3 sentences.",
)

prompt = "what can you tell me about this dude?"
image_data = Path("rick.avif").read_bytes()

result = await image_joker.run(
    user_prompt=[
        prompt,
        BinaryContent(data=image_data, media_type="image/avif")
    ]
)

result

In [18]:
print(result.response.parts[0].content)

Here's a thinking process:

1.  **Analyze User Input:**
   - User says: "A gray netherland rabbit jumps ..."
   - This is an incomplete sentence/phrase, presumably prompting me to continue a story.
   - Role: children storyteller
   - Constraint: max 3 sentences

2.  **Identify Core Task:**
   - Continue the sentence/story about a gray Netherlands rabbit jumping.
   - Keep it to 3 sentences max.
   - Target audience: children (so gentle, imaginative, simple, warm tone).

3.  **Draft - Attempt 1 (mental):**
   A gray Netherlands rabbit named Hoppy bounced across the velvety meadow, his ears flopping rhythmically in the gentle breeze. He joyfully leaped over sparkling brooks and fluffy dandelion puffs, chasing the sound of distant bird songs. With a happy twitch of his nose, he reminded everyone that even small friends can bring big dreams to life.

   Count sentences: 3. Good.

4.  **Check constraints:**
   - Max 3 sentences: ✓ (exactly 3)
   - Children storyteller tone: ✓ (imaginative,